# DeepOMAPNet — Training Notebook

End-to-end workflow: generate synthetic CITE-seq data → build graphs → train → evaluate.

In [ ]:
import sys, os, importlib, warnings
warnings.filterwarnings('ignore')

current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import torch
import numpy as np
import matplotlib.pyplot as plt

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')

## 1. Load Data (Synthetic CITE-seq)

In [ ]:
from scripts.data_provider.synthetic_citeseq import generate_citeseq_dataset, CELL_TYPE_NAMES

# Generate synthetic CITE-seq dataset (replace with real data loader if available)
dataset = generate_citeseq_dataset(n_normal=1500, n_aml=1500, seed=42)

print(f'Total cells  : {dataset.n_cells}')
print(f'RNA features : {dataset.n_genes}')
print(f'ADT proteins : {dataset.n_adts}')
print(f'AML cells    : {dataset.aml_label.sum()} ({dataset.aml_label.mean()*100:.0f}%)')
print()
print('Cell-type breakdown:')
for i, name in enumerate(CELL_TYPE_NAMES):
    count = (dataset.celltype_label == i).sum()
    print(f'  {name:<12}: {count:4d} cells ({count/dataset.n_cells*100:.1f}%)')

# Load adata_gene_train.h5ad from /projects/vanaja_lab/satya/Datasets

In [ ]:
import anndata as ad
import numpy as np
from sklearn.preprocessing import LabelEncoder

base_path = "/projects/vanaja_lab/satya/Datasets"

# ── 1. Load each modality ─────────────────────────────────────────────────────
rna_control = ad.read_h5ad(f"{base_path}/GSMControlRNA.h5ad")
rna_amla    = ad.read_h5ad(f"{base_path}/AMLARNA.h5ad")
rna_amlb    = ad.read_h5ad(f"{base_path}/AMLBRNA.h5ad")

adt_control = ad.read_h5ad(f"{base_path}/ControlADT.h5ad")
adt_amla    = ad.read_h5ad(f"{base_path}/AMLAADT.h5ad")
adt_amlb    = ad.read_h5ad(f"{base_path}/AMLBADT.h5ad")

for obj, tag in [(rna_control, 'Control'), (rna_amla, 'AMLA'), (rna_amlb, 'AMLB')]:
    obj.obs['source'] = tag
for obj, tag in [(adt_control, 'Control'), (adt_amla, 'AMLA'), (adt_amlb, 'AMLB')]:
    obj.obs['source'] = tag

# ── 2. Concatenate within each modality ──────────────────────────────────────
rna_combined = ad.concat(
    [rna_control, rna_amla, rna_amlb],
    join='inner', label='source', keys=['Control', 'AMLA', 'AMLB']
)
adt_combined = ad.concat(
    [adt_control, adt_amla, adt_amlb],
    join='inner', label='source', keys=['Control', 'AMLA', 'AMLB']
)
rna_combined.obs_names_make_unique()
adt_combined.obs_names_make_unique()

assert rna_combined.n_obs == adt_combined.n_obs, (
    f"Cell count mismatch: RNA={rna_combined.n_obs}, ADT={adt_combined.n_obs}"
)

print(f"RNA combined : {rna_combined.shape}")
print(f"ADT combined : {adt_combined.shape}")

# ── 3. AML labels — inspect the 'aml' column and convert to 0/1 ──────────────
assert 'aml' in rna_combined.obs.columns, "'aml' column not found in obs"
_aml_raw = rna_combined.obs['aml']
print(f"\n'aml' column dtype  : {_aml_raw.dtype}")
print(f"'aml' unique values : {sorted(_aml_raw.unique())}")

_aml_vals = _aml_raw.values
if np.issubdtype(_aml_raw.dtype, np.number):
    # Already numeric — cast directly
    aml_labels = _aml_vals.astype(int)
else:
    # String column: treat anything that is NOT 'Control' / '0' / 'normal'
    # (case-insensitive) as AML=1
    _non_aml = {'control', '0', 'normal', 'false', 'no'}
    aml_labels = np.array(
        [0 if str(v).strip().lower() in _non_aml else 1 for v in _aml_vals],
        dtype=int,
    )

print(f"AML labels — Control: {(aml_labels==0).sum():,}  |  AML: {(aml_labels==1).sum():,}")

# ── 4. Cell-type labels from 'Cell_type_identity' ────────────────────────────
_ct_col = 'Cell_type_identity'
assert _ct_col in rna_combined.obs.columns, f"'{_ct_col}' column not found in obs"

le = LabelEncoder()
cell_labels     = le.fit_transform(rna_combined.obs[_ct_col].fillna('Unknown').values)
num_cell_types  = int(cell_labels.max()) + 1
cell_type_names = le.classes_.tolist()

print(f"\nCell types  : {num_cell_types} unique labels")
for i, name in enumerate(cell_type_names):
    count = (cell_labels == i).sum()
    print(f"  [{i:2d}] {name:<25}: {count:6,} cells ({count/len(cell_labels)*100:.1f}%)")

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

# ── Build a joint stratification key: sample × AML status ────────────────────
# Stratifying on AML label alone lets all cells from one patient land in the
# same split, causing the val/test MSE gap.  Using sample×AML ensures every
# patient is proportionally represented in every split.
_sample_col = 'samples' if 'samples' in rna_combined.obs.columns else 'sample'
_samples    = rna_combined.obs[_sample_col].astype(str).values
strat_key   = np.array([f"{s}_{a}" for s, a in zip(_samples, aml_labels)])

# Drop any combination that has fewer than 2 cells (can't stratify on singletons)
key_counts  = pd.Series(strat_key).value_counts()
rare_keys   = set(key_counts[key_counts < 2].index)
if rare_keys:
    print(f"Collapsing {len(rare_keys)} rare strat keys to their AML label")
    strat_key = np.array(
        [str(a) if k in rare_keys else k for k, a in zip(strat_key, aml_labels)]
    )

n_cells = rna_combined.n_obs
indices = np.arange(n_cells)

# ── Stratified 80 / 10 / 10 split ────────────────────────────────────────────
idx_train, idx_temp = train_test_split(
    indices, test_size=0.20, random_state=42, stratify=strat_key
)
idx_val, idx_test = train_test_split(
    idx_temp, test_size=0.50, random_state=42, stratify=strat_key[idx_temp]
)

# ── Slice both modalities ─────────────────────────────────────────────────────
rna_train = rna_combined[idx_train].copy()
rna_val   = rna_combined[idx_val].copy()
rna_test  = rna_combined[idx_test].copy()

adt_train = adt_combined[idx_train].copy()
adt_val   = adt_combined[idx_val].copy()
adt_test  = adt_combined[idx_test].copy()

# ── Slice labels ──────────────────────────────────────────────────────────────
aml_train = aml_labels[idx_train]
aml_val   = aml_labels[idx_val]
aml_test  = aml_labels[idx_test]

ct_train = cell_labels[idx_train] if cell_labels is not None else None
ct_val   = cell_labels[idx_val]   if cell_labels is not None else None
ct_test  = cell_labels[idx_test]  if cell_labels is not None else None

# ── Sanity-check ──────────────────────────────────────────────────────────────
print("=" * 60)
print(f"{'Split':<10} {'RNA shape':>15} {'ADT shape':>15}")
print("=" * 60)
for tag, r, a in [("Train", rna_train, adt_train),
                   ("Val",   rna_val,   adt_val),
                   ("Test",  rna_test,  adt_test)]:
    print(f"{tag:<10} {str(r.shape):>15} {str(a.shape):>15}")
print("=" * 60)

print("\nAML distribution per split:")
for tag, lbl in [("Train", aml_train), ("Val", aml_val), ("Test", aml_test)]:
    print(f"  {tag:<6}: Control={( lbl==0).sum():,}  AML={(lbl==1).sum():,}  "
          f"(AML ratio = {(lbl==1).mean():.2%})")

print("\nSample distribution per split:")
for tag, idx in [("Train", idx_train), ("Val", idx_val), ("Test", idx_test)]:
    samples_in_split = rna_combined.obs[_sample_col].iloc[idx].value_counts()
    print(f"  {tag}: {dict(samples_in_split)}")

assert len(set(idx_train) & set(idx_val))  == 0, "Train/Val overlap!"
assert len(set(idx_train) & set(idx_test)) == 0, "Train/Test overlap!"
assert len(set(idx_val)   & set(idx_test)) == 0, "Val/Test overlap!"
print("\nNo index overlap ✓")

## 2. Build PyTorch Geometric Graphs

The trainer handles the 80/10/10 train/val/test split internally via node masks.
We pass the full combined RNA and ADT objects — no manual splitting needed.

In [ ]:
import scanpy as sc
import scipy.sparse as sp

# ── 1. CLR-normalize ADT counts ───────────────────────────────────────────────
# CLR (Centered Log-Ratio) is the standard normalization for CITE-seq protein
# data. For each cell: CLR(x_j) = log(x_j + 1) - mean_j( log(x_j + 1) )
# This removes the cell-size effect while preserving per-protein variation.
# Applied to train/val/test using the same per-cell formula (no leakage since
# CLR is computed independently per cell, not across cells).
def _clr_normalize(adata, label):
    X = adata.X.toarray() if sp.issparse(adata.X) else np.array(adata.X, dtype=np.float32)
    if X.min() < 0:
        print(f"{label}: negative values detected — skipping CLR (already normalized?)")
        return
    log1p_X = np.log1p(X)                            # log(x + 1)
    clr_X   = log1p_X - log1p_X.mean(axis=1, keepdims=True)  # subtract per-cell mean
    adata.X = clr_X.astype(np.float32)
    print(f"{label}: CLR applied  range=[{clr_X.min():.2f}, {clr_X.max():.2f}]  "
          f"mean={clr_X.mean():.4f}")

_clr_normalize(adt_train, "adt_train")
_clr_normalize(adt_val,   "adt_val")
_clr_normalize(adt_test,  "adt_test")

# ── 2. Log-normalize RNA counts ───────────────────────────────────────────────
def _log_normalize(adata, label):
    X = adata.X.toarray() if sp.issparse(adata.X) else np.array(adata.X)
    if X.max() > 50:
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        _max = adata.X.data.max() if sp.issparse(adata.X) else adata.X.max()
        print(f"{label}: log-normalized  (new max={_max:.2f})")
    else:
        print(f"{label}: already normalized, skipping (max={X.max():.2f})")

_log_normalize(rna_train, "rna_train")
_log_normalize(rna_val,   "rna_val")
_log_normalize(rna_test,  "rna_test")

# ── 3. HVG selection on log-normalized train only ────────────────────────────
N_HVG = 2000
sc.pp.highly_variable_genes(rna_train, n_top_genes=N_HVG, flavor="seurat_v3", span=0.3)
hvg_genes = rna_train.var_names[rna_train.var["highly_variable"]].tolist()
print(f"\nHVGs selected : {len(hvg_genes)} / {rna_train.n_vars} genes")

# ── 4. Subset RNA splits to HVGs ─────────────────────────────────────────────
rna_train = rna_train[:, hvg_genes].copy()
rna_val   = rna_val[:,   hvg_genes].copy()
rna_test  = rna_test[:,  hvg_genes].copy()

print(f"\nFinal shapes:")
print(f"  RNA  — Train:{rna_train.shape}  Val:{rna_val.shape}  Test:{rna_test.shape}")
print(f"  ADT  — Train:{adt_train.shape}  Val:{adt_val.shape}  Test:{adt_test.shape}")

In [ ]:
from scripts.data_provider.graph_data_builder import build_pyg_data

print("Building RNA graph...")
rna_pyg = build_pyg_data(rna_train)
print(f"  {rna_pyg}")

print("Building ADT graph...")
adt_pyg = build_pyg_data(adt_train)
print(f"  {adt_pyg}")

assert rna_pyg.num_nodes == adt_pyg.num_nodes, "Node count mismatch!"
print(f"\nGraphs built — {rna_pyg.num_nodes} cells, "
      f"{rna_pyg.num_edges} RNA edges, {adt_pyg.num_edges} ADT edges")

In [ ]:
import importlib, logging, sys
import scripts.trainer.gat_trainer as _gat_module
importlib.reload(_gat_module)
from scripts.trainer.gat_trainer import train_gat_transformer_fusion

# ── Route training logs to notebook stdout ────────────────────────────────────
_log = logging.getLogger("scripts.trainer.gat_trainer")
_log.handlers.clear()
_log.setLevel(logging.INFO)
_h = logging.StreamHandler(sys.stdout)
_h.setFormatter(logging.Formatter("%(asctime)s | %(message)s", datefmt="%H:%M:%S"))
_log.addHandler(_h)
_log.propagate = False

training_config = dict(
    epochs=1000,
    seed=42,
    learning_rate=1e-3,
    weight_decay=1e-4,
    dropout_rate=0.3,
    hidden_channels=256,       # was 64  — enough capacity for 2000-gene input
    num_heads=8,               # was 4
    num_attention_heads=8,     # was 4
    num_layers=3,              # was 2
    use_mixed_precision=torch.cuda.is_available(),
    early_stopping_patience=20,  # was 10 — give the larger model more time
)

result = train_gat_transformer_fusion(
    rna_data=rna_pyg,
    adt_data=adt_pyg,
    rna_anndata=rna_train,
    adt_anndata=adt_train,
    aml_labels=torch.tensor(aml_train, dtype=torch.float32),
    celltype_labels=torch.tensor(ct_train, dtype=torch.long),
    num_cell_types=num_cell_types,
    celltype_weight=0.5,
    stratify_labels=aml_train,
    **training_config,
)

trained_model         = result.model
rna_data_with_masks   = result.rna_data
adt_data_with_masks   = result.adt_data
training_history      = result.history
adt_mean              = result.normalization.adt_mean
adt_std               = result.normalization.adt_std
node_degrees_rna      = result.graph_stats.node_degrees_rna
node_degrees_adt      = result.graph_stats.node_degrees_adt
clustering_coeffs_rna = result.graph_stats.clustering_coeffs_rna
clustering_coeffs_adt = result.graph_stats.clustering_coeffs_adt

print(f"\nBest val  R²: {max(training_history['val_R2']):.4f}")
print(f"Final test R²: {training_history['test_R2'][-1]:.4f}")

from scripts.trainer.gat_trainer import train_gat_transformer_fusion

training_config = dict(
    epochs=100,
    seed=42,
    learning_rate=1e-3,
    weight_decay=1e-4,
    dropout_rate=0.4,
    hidden_channels=64,
    num_heads=4,
    num_attention_heads=4,
    num_layers=2,
    use_mixed_precision=torch.cuda.is_available(),
    early_stopping_patience=10,
)

(
    trained_model,
    rna_data_with_masks,
    adt_data_with_masks,
    training_history,
    adt_mean, adt_std,
    node_degrees_rna, node_degrees_adt,
    clustering_coeffs_rna, clustering_coeffs_adt,
) = train_gat_transformer_fusion(
    rna_data=rna_pyg,
    adt_data=adt_pyg,
    aml_labels=aml_labels,
    celltype_labels=cell_labels,        # None if not available
    num_cell_types=num_cell_types,      # None if not available
    celltype_weight=0.5,
    stratify_labels=aml_labels,         # keeps AML ratio balanced across splits
    **training_config,
)

print(f"\nBest val  R²: {max(training_history['val_R2']):.4f}")
print(f"Final test R²: {training_history['test_R2'][-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(training_history['train_loss'])
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch')

axes[1].plot(training_history['val_MSE'],  label='Val')
axes[1].plot(training_history['test_MSE'], label='Test')
axes[1].set_title('ADT MSE'); axes[1].set_xlabel('Epoch'); axes[1].legend()

axes[2].plot(training_history['val_R2'],  label='Val')
axes[2].plot(training_history['test_R2'], label='Test')
axes[2].set_title('ADT R²'); axes[2].set_xlabel('Epoch'); axes[2].legend()

plt.tight_layout()
plt.show()

## 6. Evaluate on Held-Out Test Set

In [ ]:
import scipy.sparse as sp
from sklearn.metrics import r2_score, mean_squared_error, accuracy_score, roc_auc_score

device = next(trained_model.parameters()).device

rna_inf = rna_data_with_masks.to(device)
adt_inf = adt_data_with_masks.to(device)
test_mask    = rna_inf.test_mask          # bool mask over rna_train nodes
adt_mean_dev = adt_mean.to(device)
adt_std_dev  = adt_std.to(device)

trained_model.eval()
with torch.no_grad():
    adt_pred, aml_pred, fused_emb = trained_model(
        x=rna_inf.x,
        edge_index_rna=rna_inf.edge_index,
        edge_index_adt=adt_inf.edge_index,
        node_degrees_rna=node_degrees_rna.to(device),
        node_degrees_adt=node_degrees_adt.to(device),
        clustering_coeffs_rna=clustering_coeffs_rna.to(device),
        clustering_coeffs_adt=clustering_coeffs_adt.to(device),
    )

# Denormalize predictions and targets back to CLR space
test_mask_cpu = test_mask.cpu().numpy()
adt_pred_np = (adt_pred[test_mask] * adt_std_dev + adt_mean_dev).float().cpu().numpy()
adt_true_np = (adt_inf.x[test_mask] * adt_std_dev + adt_mean_dev).float().cpu().numpy()

r2  = r2_score(adt_true_np.ravel(), adt_pred_np.ravel())
mse = mean_squared_error(adt_true_np, adt_pred_np)

# AML labels: test_mask indexes into rna_train nodes, so use aml_train
aml_probs  = torch.sigmoid(aml_pred[test_mask]).cpu().numpy().flatten()
aml_binary = (aml_probs > 0.5).astype(int)
aml_true   = aml_train[test_mask_cpu]   # same length as rna_data_with_masks

acc = accuracy_score(aml_true, aml_binary)
auc = roc_auc_score(aml_true, aml_probs)

print("=== Internal Test Split Results ===")
print(f"ADT  R²      : {r2:.4f}")
print(f"ADT  MSE     : {mse:.4f}")
print(f"AML  Accuracy: {acc:.4f}")
print(f"AML  AUC-ROC : {auc:.4f}")
print(f"\n(test nodes: {test_mask_cpu.sum():,} / {len(test_mask_cpu):,})")

from sklearn.metrics import r2_score, mean_squared_error, accuracy_score, roc_auc_score

device = next(trained_model.parameters()).device

# Move everything the model needs to the same device
rna_inf = rna_data_with_masks.to(device)
adt_inf = adt_data_with_masks.to(device)
test_mask    = rna_inf.test_mask
adt_mean_dev = adt_mean.to(device)
adt_std_dev  = adt_std.to(device)
nd_rna = node_degrees_rna.to(device)
nd_adt = node_degrees_adt.to(device)
cc_rna = clustering_coeffs_rna.to(device)
cc_adt = clustering_coeffs_adt.to(device)

trained_model.eval()
with torch.no_grad():
    adt_pred, aml_pred, fused_emb = trained_model(
        x=rna_inf.x,
        edge_index_rna=rna_inf.edge_index,
        edge_index_adt=adt_inf.edge_index,
        node_degrees_rna=nd_rna,
        node_degrees_adt=nd_adt,
        clustering_coeffs_rna=cc_rna,
        clustering_coeffs_adt=cc_adt,
    )

# Denormalise ADT predictions
adt_pred_np = (adt_pred[test_mask] * adt_std_dev + adt_mean_dev).cpu().numpy()
adt_true_np = (adt_inf.x[test_mask]  * adt_std_dev + adt_mean_dev).cpu().numpy()

# ADT regression metrics
r2  = r2_score(adt_true_np.ravel(), adt_pred_np.ravel())
mse = mean_squared_error(adt_true_np, adt_pred_np)

# AML classification metrics
aml_probs  = torch.sigmoid(aml_pred[test_mask]).cpu().numpy().flatten()
aml_binary = (aml_probs > 0.5).astype(int)
aml_true   = torch.tensor(aml_labels, dtype=torch.float32)[test_mask.cpu()].numpy().astype(int)
acc = accuracy_score(aml_true, aml_binary)
auc = roc_auc_score(aml_true, aml_probs)

print("=== Held-out Test Results ===")
print(f"ADT  R²      : {r2:.4f}")
print(f"ADT  MSE     : {mse:.4f}")
print(f"AML  Accuracy: {acc:.4f}")
print(f"AML  AUC-ROC : {auc:.4f}")

In [ ]:
from scipy.stats import pearsonr

protein_names = list(adt_combined.var_names)
n_proteins    = adt_pred_np.shape[1]

correlations = [pearsonr(adt_true_np[:, p], adt_pred_np[:, p])[0] for p in range(n_proteins)]

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(protein_names, correlations)
ax.axhline(0, color='k', linewidth=0.5)
ax.set_ylabel('Pearson r')
ax.set_title('Per-Protein Prediction Correlation (Test Set)')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

best_idx  = int(np.argmax(correlations))
worst_idx = int(np.argmin(correlations))
print(f"Mean correlation : {np.mean(correlations):.4f}")
print(f"Best protein     : {protein_names[best_idx]}  (r={correlations[best_idx]:.4f})")
print(f"Worst protein    : {protein_names[worst_idx]}  (r={correlations[worst_idx]:.4f})")

In [ ]:
from sklearn.metrics import classification_report
import matplotlib.patches as mpatches

# ── Labels for test nodes ─────────────────────────────────────────────────────
ct_true       = ct_train[test_mask_cpu]
aml_true_mask = aml_train[test_mask_cpu]

# ── Cell-type predictions ─────────────────────────────────────────────────────
with torch.no_grad():
    ct_logits = trained_model.predict_celltypes(fused_emb[test_mask].float().to(device))
ct_pred = ct_logits.argmax(dim=-1).cpu().numpy()

# ── AML predictions ───────────────────────────────────────────────────────────
aml_pred_binary = (torch.sigmoid(aml_pred[test_mask]).cpu().numpy().flatten() > 0.5).astype(int)

print("Cell-type classification report:")
print(classification_report(ct_true, ct_pred, target_names=cell_type_names, zero_division=0))
print("AML classification report:")
print(classification_report(aml_true_mask, aml_pred_binary,
                             target_names=["Control", "AML"], zero_division=0))

# ── Colour palettes ───────────────────────────────────────────────────────────
n_ct     = num_cell_types
ct_cmap  = plt.cm.get_cmap("tab20", n_ct)
aml_cmap = {0: "#4878CF", 1: "#D65F5F"}

# ── Figure: cell type (top row) + AML (bottom row) ───────────────────────────
# Each row has: [scatter | legend-only axes]  x2 panels
# Legend is placed in a dedicated axes to avoid overlapping the scatter.
fig = plt.figure(figsize=(18, 14))

# Column widths: scatter(4) | legend(1) | gap(0.3) | scatter(4) | legend(1)
gs = fig.add_gridspec(2, 5, width_ratios=[4, 1, 0.3, 4, 1],
                       hspace=0.35, wspace=0.05)

def _scatter_ct(ax, labels, title):
    colors = [ct_cmap(l) for l in labels]
    ax.scatter(umap_coords[:, 0], umap_coords[:, 1],
               c=colors, s=1.5, rasterized=True, linewidths=0)
    ax.set_title(title, fontsize=11, pad=6)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel("UMAP 1", fontsize=8); ax.set_ylabel("UMAP 2", fontsize=8)

def _scatter_aml(ax, labels, title):
    colors = [aml_cmap[l] for l in labels]
    ax.scatter(umap_coords[:, 0], umap_coords[:, 1],
               c=colors, s=1.5, rasterized=True, linewidths=0)
    ax.set_title(title, fontsize=11, pad=6)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel("UMAP 1", fontsize=8); ax.set_ylabel("UMAP 2", fontsize=8)

def _legend_ct(ax):
    ax.axis("off")
    handles = [mpatches.Patch(color=ct_cmap(i), label=cell_type_names[i])
               for i in range(n_ct)]
    ax.legend(handles=handles, loc="center left", fontsize=7,
              framealpha=0.8, handlelength=1.2, borderpad=0.8,
              title="Cell Type", title_fontsize=8)

def _legend_aml(ax):
    ax.axis("off")
    handles = [mpatches.Patch(color=aml_cmap[v], label=lbl)
               for v, lbl in [(0, "Control"), (1, "AML")]]
    ax.legend(handles=handles, loc="center left", fontsize=9,
              framealpha=0.8, handlelength=1.2, borderpad=0.8,
              title="AML Status", title_fontsize=9)

# Row 0 — Cell Type
ax00 = fig.add_subplot(gs[0, 0]); _scatter_ct(ax00, ct_true,  "Cell Type — True")
ax01 = fig.add_subplot(gs[0, 1]); _legend_ct(ax01)
ax03 = fig.add_subplot(gs[0, 3]); _scatter_ct(ax03, ct_pred,  "Cell Type — Predicted")
ax04 = fig.add_subplot(gs[0, 4]); _legend_ct(ax04)

# Row 1 — AML
ax10 = fig.add_subplot(gs[1, 0]); _scatter_aml(ax10, aml_true_mask,   "AML Status — True")
ax11 = fig.add_subplot(gs[1, 1]); _legend_aml(ax11)
ax13 = fig.add_subplot(gs[1, 3]); _scatter_aml(ax13, aml_pred_binary, "AML Status — Predicted")
ax14 = fig.add_subplot(gs[1, 4]); _legend_aml(ax14)

# hide gap columns
for r in range(2):
    fig.add_subplot(gs[r, 2]).set_visible(False)

fig.suptitle("Cell Type & AML: True vs Predicted (UMAP of Fused Embeddings)",
             fontsize=13, y=1.01)
plt.savefig("celltype_aml_umap.pdf", bbox_inches="tight", dpi=150)
plt.show()
print("Saved → celltype_aml_umap.pdf")

In [ ]:
from sklearn.metrics import classification_report
import matplotlib.patches as mpatches

# ── UMAP coords already computed above (umap_coords, fused_test_np) ──────────
# ── Labels for test nodes ─────────────────────────────────────────────────────
ct_true  = ct_train[test_mask_cpu]          # true cell-type indices
aml_true_mask = aml_train[test_mask_cpu]    # true AML labels (0/1)

# ── Cell-type predictions ─────────────────────────────────────────────────────
with torch.no_grad():
    ct_logits = trained_model.predict_celltypes(fused_emb[test_mask].float().to(device))
ct_pred = ct_logits.argmax(dim=-1).cpu().numpy()

# ── AML predictions ───────────────────────────────────────────────────────────
aml_pred_binary = (torch.sigmoid(aml_pred[test_mask]).cpu().numpy().flatten() > 0.5).astype(int)

print("Cell-type classification report:")
print(classification_report(ct_true, ct_pred, target_names=cell_type_names, zero_division=0))
print("AML classification report:")
from sklearn.metrics import classification_report as cr
print(cr(aml_true_mask, aml_pred_binary, target_names=["Control", "AML"], zero_division=0))

# ── Colour palettes ───────────────────────────────────────────────────────────
n_ct      = num_cell_types
ct_cmap   = plt.cm.get_cmap("tab20", n_ct)
aml_cmap  = {0: "#4878CF", 1: "#D65F5F"}   # blue=Control, red=AML

fig, axes = plt.subplots(2, 2, figsize=(14, 12), constrained_layout=True)

panels = [
    # (ax,  labels,          title,                        kind)
    (axes[0, 0], ct_true,         "Cell Type — True",           "ct"),
    (axes[0, 1], ct_pred,         "Cell Type — Predicted",      "ct"),
    (axes[1, 0], aml_true_mask,   "AML Status — True",          "aml"),
    (axes[1, 1], aml_pred_binary, "AML Status — Predicted",     "aml"),
]

for ax, labels, title, kind in panels:
    if kind == "ct":
        colors = [ct_cmap(l) for l in labels]
        ax.scatter(umap_coords[:, 0], umap_coords[:, 1],
                   c=colors, s=1.5, rasterized=True, linewidths=0)
        handles = [mpatches.Patch(color=ct_cmap(i), label=cell_type_names[i])
                   for i in range(n_ct)]
        ax.legend(handles=handles, fontsize=6, loc="best",
                  markerscale=3, framealpha=0.7, ncol=2)
    else:
        colors = [aml_cmap[l] for l in labels]
        ax.scatter(umap_coords[:, 0], umap_coords[:, 1],
                   c=colors, s=1.5, rasterized=True, linewidths=0)
        handles = [mpatches.Patch(color=aml_cmap[v], label=lbl)
                   for v, lbl in [(0, "Control"), (1, "AML")]]
        ax.legend(handles=handles, fontsize=8, loc="best",
                  markerscale=4, framealpha=0.7)

    ax.set_title(title, fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel("UMAP 1", fontsize=8)
    ax.set_ylabel("UMAP 2", fontsize=8)

fig.suptitle("Cell Type & AML Predictions vs Ground Truth (UMAP of Fused Embeddings)",
             fontsize=13)
plt.savefig("celltype_aml_umap.pdf", bbox_inches="tight", dpi=150)
plt.show()
print("Saved → celltype_aml_umap.pdf")

from scipy.stats import pearsonr

protein_names = list(adt_combined.var_names)
n_proteins    = adt_pred_np.shape[1]

correlations = [
    pearsonr(adt_true_np[:, p], adt_pred_np[:, p])[0]
    for p in range(n_proteins)
]

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(protein_names, correlations)
ax.axhline(0, color='k', linewidth=0.5)
ax.set_ylabel("Pearson r")
ax.set_title("Per-Protein Prediction Correlation (Test Set)")
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

best_idx = int(np.argmax(correlations))
print(f"Mean correlation : {np.mean(correlations):.4f}")
print(f"Best protein     : {protein_names[best_idx]} ({correlations[best_idx]:.4f}"
      f"  worst: {protein_names[int(np.argmin(correlations))]} ({min(correlations):.4f})")

In [ ]:
torch.save(trained_model.state_dict(), 'DeepOMAPNet_weights.pth')
print('Model saved to DeepOMAPNet_weights.pth')